# Check_FabriBuddy_Version

Compares the FabriBuddy release installed in `FabriBuddy_DB` with the latest one on GitHub.
To update, import the latest `Deploy_FabriBuddy` notebook and run it.

[How to update](https://zan-tavcar.github.io/msfabric-fabribuddy/update/)

In [ ]:
sql_database_name = "FabriBuddy_DB"
repository = "zan-tavcar/msfabric-fabribuddy"
github_token = ""   # only while the repository is private

In [ ]:
# FabriBuddy_DB, before the FabriBuddy package is installed. As whoever runs this notebook.
import json, struct, subprocess, sys, time
import requests

FABRIC_API = "https://api.fabric.microsoft.com/v1"
WORKSPACE_ID = notebookutils.runtime.context["currentWorkspaceId"]


def fabric(method, path, body=None, wait=True):
    """The Fabric REST API as you; waits for long-running operations."""
    headers = {"Authorization": f"Bearer {notebookutils.credentials.getToken('pbi')}"}
    r = requests.request(method, f"{FABRIC_API}{path}", headers=headers, json=body, timeout=120)
    if r.status_code >= 400:
        raise RuntimeError(f"{method} {path} failed ({r.status_code}): {r.text[:400]}")
    if r.status_code == 202 and wait and r.headers.get("Location"):
        location, state = r.headers["Location"], None
        for _ in range(200):
            time.sleep(min(int(r.headers.get("Retry-After", 3) or 3), 10))
            r = requests.get(location, headers=headers, timeout=60)
            state = (r.json() if r.text else {}).get("status")
            if state in ("Succeeded", "Failed", "Undefined"):
                break
        if state != "Succeeded":
            raise RuntimeError(f"{method} {path} did not succeed: {r.text[:400]}")
        result = requests.get(location + "/result", headers=headers, timeout=60)
        return result.json() if result.status_code == 200 and result.text else {}
    return r.json() if r.text else {}


def find_sql_database(name):
    """(item id, server, database) of the SQL database called `name` here, or (None, "", "")."""
    for item in fabric("GET", f"/workspaces/{WORKSPACE_ID}/sqlDatabases").get("value", []):
        if item["displayName"].lower() == name.lower():
            props = fabric("GET", f"/workspaces/{WORKSPACE_ID}/sqlDatabases/{item['id']}").get("properties") or {}
            return item["id"], (props.get("serverFqdn") or "").split(",")[0], props.get("databaseName") or ""
    return None, "", ""


def sql_token():
    return notebookutils.credentials.getToken("https://database.windows.net/")


def sql_connect(server, database):
    import pyodbc

    raw = sql_token().encode("utf-16-le")
    return pyodbc.connect(
        f"Driver={{ODBC Driver 18 for SQL Server}};Server={server},1433;Database={database};"
        "Encrypt=yes;TrustServerCertificate=no",
        attrs_before={1256: struct.pack(f"<I{len(raw)}s", len(raw), raw)},
        timeout=30,
    )


def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", *args])

In [ ]:
_, server, database = find_sql_database(sql_database_name)
with sql_connect(server, database) as conn:
    row = conn.cursor().execute("SELECT version, saved_at FROM dbo.fabribuddy_package").fetchone()
installed = row[0] if row else None

headers = {"Authorization": f"token {github_token}"} if github_token else {}
latest = requests.get(f"https://raw.githubusercontent.com/{repository}/main/data/current_fabribuddy_version.json",
                      headers=headers, timeout=60).json()["version"]

print(f"Installed: {installed or 'none'}   Latest: {latest}")
print("FabriBuddy is up to date." if installed == latest else
      f"An update is available. See https://zan-tavcar.github.io/msfabric-fabribuddy/update/")